# Scikit-learn 기초

Scikit-learn은 Python ML 라이브러리의 표준입니다.

## 학습 목표
- 지도학습 (회귀, 분류)
- 비지도학습 (클러스터링)
- 모델 평가 방법
- 특성 공학 기초

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print("Scikit-learn 준비 완료!")

## 1. 분류: 붓꽃 분류 (Iris)

In [ ]:
from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

# 데이터 로딩
iris = load_iris()
X = iris.data
y = iris.target

print(f"특성: {iris.feature_names}")
print(f"클래스: {iris.target_names}")
print(f"데이터 형태: {X.shape}")

In [ ]:
# 학습/테스트 분할
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"훈련 데이터: {X_train.shape[0]}개")
print(f"테스트 데이터: {X_test.shape[0]}개")

In [ ]:
# 모델 학습
model = DecisionTreeClassifier(random_state=42)
model.fit(X_train, y_train)

# 예측
y_pred = model.predict(X_test)

# 평가
print(f"정확도: {accuracy_score(y_test, y_pred):.4f}")
print("\n분류 리포트:")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

In [ ]:
# 혼동 행렬
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=iris.target_names,
            yticklabels=iris.target_names)
plt.xlabel('예측')
plt.ylabel('실제')
plt.title('혼동 행렬')
plt.show()

## 2. 여러 분류 알고리즘 비교

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

# 모델 딕셔너리
models = {
    '로지스틱 회귀': LogisticRegression(max_iter=200),
    '결정트리': DecisionTreeClassifier(),
    '랜덤 포레스트': RandomForestClassifier(),
    'SVM': SVC(),
    'KNN': KNeighborsClassifier()
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    score = model.score(X_test, y_test)
    results[name] = score
    print(f"{name}: {score:.4f}")

In [ ]:
# 모델 비교 시각화
plt.figure(figsize=(10, 5))
plt.bar(results.keys(), results.values())
plt.ylabel('정확도')
plt.title('분류 모델 비교')
plt.ylim(0.8, 1.0)
plt.xticks(rotation=45)
plt.show()

## 3. 회귀: 보스턴 주택 가격

In [ ]:
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# 회귀 데이터 생성
X_reg, y_reg = make_regression(n_samples=200, n_features=1, noise=20, random_state=42)

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

# 모델 학습
reg_model = LinearRegression()
reg_model.fit(X_train_r, y_train_r)

# 예측
y_pred_r = reg_model.predict(X_test_r)

print(f"R² 점수: {r2_score(y_test_r, y_pred_r):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test_r, y_pred_r)):.4f}")

In [ ]:
# 회귀 시각화
plt.figure(figsize=(10, 5))
plt.scatter(X_test_r, y_test_r, label='실제값', alpha=0.6)
plt.plot(X_test_r, y_pred_r, color='red', label='예측값')
plt.xlabel('X')
plt.ylabel('y')
plt.title('선형 회귀')
plt.legend()
plt.show()

## 4. 클러스터링: K-Means

In [ ]:
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs

# 클러스터 데이터 생성
X_cluster, y_cluster = make_blobs(n_samples=300, centers=4, cluster_std=0.60, random_state=42)

# K-Means 학습
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
labels = kmeans.fit_predict(X_cluster)

plt.figure(figsize=(10, 5))
plt.scatter(X_cluster[:, 0], X_cluster[:, 1], c=labels, cmap='viridis', alpha=0.6)
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
            s=300, c='red', marker='X', label='센트로이드')
plt.title('K-Means 클러스터링')
plt.legend()
plt.show()

In [ ]:
# 엘보우 방법 (최적 K 찾기)
inertias = []
K_range = range(1, 10)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_cluster)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(K_range, inertias, 'bo-')
plt.xlabel('클러스터 수 (K)')
plt.ylabel('관성 (Inertia)')
plt.title('엘보우 방법')
plt.show()

## 5. 교차 검증

In [ ]:
from sklearn.model_selection import cross_val_score

# 5-fold 교차 검증
cv_scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')

print(f"교차 검증 점수: {cv_scores}")
print(f"평균: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")

## 6. 특성 중요도

In [ ]:
# 랜덤 포레스트 특성 중요도
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(8, 5))
plt.bar(range(X.shape[1]), importances[indices])
plt.xticks(range(X.shape[1]), [iris.feature_names[i] for i in indices], rotation=45)
plt.ylabel('중요도')
plt.title('특성 중요도 (랜덤 포레스트)')
plt.tight_layout()
plt.show()

## 연습 문제

1. digits 데이터셋으로 숫자 분류를 수행하세요.
2. 최적의 K값을 찾아 클러스터링하세요.
3. 회귀 모델에서 Ridge, Lasso를 비교하세요.